### 데이터 불러오기

In [19]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"data\police_time.csv",encoding='CP949')
df.head()

,범죄대분류,범죄중분류,0시00분-02시59분,03시00분-05시59분,06시00분-08시59분,09시00분-11시59분,12시00분-14시59분,15시00분-17시59분,18시00분-20시59분,21시00분-23시59분,미상
0,강력범죄,살인기수,15,21,20,28,38,33,43,49,50
1,강력범죄,살인미수등,33,50,29,44,46,60,66,82,72
2,강력범죄,강도,105,131,44,59,68,72,81,107,131
3,강력범죄,강간,630,885,417,293,325,311,425,719,1305
4,강력범죄,유사강간,102,110,57,62,60,55,66,89,181


### 데이터 전처리
- 1. 특정 범죄 항목 제외 (데이터 필터링)
- 2. '미상' 데이터를 다른 시간대(8개 구간)에 n분의 1로 배분
- 3. '미상' 컬럼은 이제 필요 없으므로 삭제
- 4. 데이터 멜트 (Melt)
- 5. 저장

In [28]:
# 특정 범죄 항목 제외 (데이터 필터링)
# 분석 목적에 맞지 않거나 제외할 범죄 소분류 리스트를 정의
delete_list = ['유사강간', '기타 강간 강제추행등', '강제추행', '방화', '약취 유인', '폭력행위등', '공갈', '손괴', '체포 감금']
df_filtered = df[~df['범죄중분류'].isin(delete_list)].copy()

# '미상' 데이터를 다른 시간대에 n분의 1로 배분
# 시간대 컬럼 리스트 (0시~23시까지 총 8개 구간)
time_cols = [
    '0시00분-02시59분', '03시00분-05시59분', '06시00분-08시59분', 
    '09시00분-11시59분', '12시00분-14시59분', '15시00분-17시59분', 
    '18시00분-20시59분', '21시00분-23시59분'
]

# '미상' 값을 8로 나누어 각 시간대에 더해줌
n = len(time_cols)
for col in time_cols:
    df_filtered[col] = (df_filtered[col] + (df_filtered['미상'] / n)).round(2)

# '미상' 컬럼은 이제 필요 없으므로 삭제
df_filtered = df_filtered.drop(columns=['미상'])

# 데이터 멜트 (Melt)
# 범죄대분류, 범죄중분류를 고정하고 시간대 컬럼들을 행으로 내림
df_melted = df_filtered.melt(
    id_vars=['범죄대분류', '범죄중분류'], 
    var_name='시간대', 
    value_name='범죄건수'
)

# 결과 확인
display(df_melted)

# 전처리된 파일 저장
df_melted.to_csv('police_time_fix.csv', index=False, encoding='utf-8-sig')

,범죄대분류,범죄중분류,시간대,범죄건수
0,강력범죄,살인기수,0시00분-02시59분,21.25
1,강력범죄,살인미수등,0시00분-02시59분,42.00
2,강력범죄,강도,0시00분-02시59분,121.38
3,강력범죄,강간,0시00분-02시59분,793.12
4,절도범죄,절도,0시00분-02시59분,15786.75
...,...,...,...,...
67,절도범죄,절도,21시00분-23시59분,23266.75
68,폭력범죄,상해,21시00분-23시59분,8100.88
69,폭력범죄,폭행,21시00분-23시59분,36436.75
70,폭력범죄,협박,21시00분-23시59분,3857.50
